# Mule Networks & AML

- [Graph Analytics](#Graph-Analytics)
  - [Representations](#Representations)
  - [Centrality measures](#Centrality-measures)
  - [Community detection](#Community-detection)
  - [PageRank, worked by hand](#PageRank,-worked-by-hand)
  - [Relevance to fraud detection](#Relevance-to-fraud-detection)
  - [Worked example: transaction network (NetworkX)](#Worked-example:-transaction-network-(NetworkX))
  - [Clustering coefficient and density](#Clustering-coefficient-and-density)
  - [Link prediction](#Link-prediction)
  - [Distance to known-bad accounts](#Distance-to-known-bad-accounts)
  - [Graph embeddings and features for ML models](#Graph-embeddings-and-features-for-ML-models)
  - [Graph neural networks](#Graph-neural-networks)
- [AML Monitoring and Mule Networks](#AML-Monitoring-and-Mule-Networks)
  - [What AML monitoring does](#What-AML-monitoring-does)
  - [Mule accounts](#Mule-accounts)
  - [Why one bank is not enough](#Why-one-bank-is-not-enough)
- [Entity Resolution at Scale (TF-IDF / Cosine Similarity + Blocking)](#Entity-Resolution-at-Scale-(TF-IDF-/-Cosine-Similarity-+-Blocking))
- [Blogs](#Blogs)

### Graph Analytics

A graph represents entities as nodes and relationships as edges. It is useful when the connections between data points carry signal that a flat tabular row cannot capture. In fraud, rings show up as unusually dense clusters or high-centrality nodes in a graph of shared identifiers (phone numbers, devices, payment methods across accounts that look unrelated in a flat table).

#### Representations

Toy graph with 5 accounts, where an edge means two accounts share a payment method or device:
```
edges: A-B, A-C, B-C, C-D, D-E
```
A, B and C form a tight triangle (all pairwise connected), C also bridges to D, and D bridges to the otherwise isolated E.

Adjacency matrix (symmetric for an undirected graph, 1 if an edge exists):
```
     A  B  C  D  E
  A  0  1  1  0  0
  B  1  0  1  0  0
  C  1  1  0  1  0
  D  0  0  1  0  1
  E  0  0  0  1  0
```
Adjacency list, which is more compact for sparse graphs (most real graphs are sparse, since most pairs of nodes are not connected): `{A: [B,C], B: [A,C], C: [A,B,D], D: [C,E], E: [D]}`.

In [1]:
import networkx as nx

G = nx.Graph()
G.add_edges_from([("A","B"), ("A","C"), ("B","C"), ("C","D"), ("D","E")])

print("adjacency matrix:\n", nx.to_numpy_array(G, nodelist=["A","B","C","D","E"]))
print("\nadjacency list:", {n: list(G.neighbors(n)) for n in G.nodes})

adjacency matrix:
 [[0. 1. 1. 0. 0.]
 [1. 0. 1. 0. 0.]
 [1. 1. 0. 1. 0.]
 [0. 0. 1. 0. 1.]
 [0. 0. 0. 1. 0.]]

adjacency list: {'A': ['B', 'C'], 'B': ['A', 'C'], 'C': ['A', 'B', 'D'], 'D': ['C', 'E'], 'E': ['D']}


#### Centrality measures

**Degree centrality** is the number of edges touching a node. On the toy graph: degree(A)=2, degree(B)=2, degree(C)=3, degree(D)=2, degree(E)=1. C has the most connections, which is the first and cheapest signal for "this account looks like a hub".

**Betweenness centrality** measures how often a node lies on the shortest path between other pairs of nodes. A node with modest degree can have high betweenness if it is the only bridge between two parts of the graph. In the toy graph D has low degree (2), but every path from E to the rest of the graph passes through D. That bridging role is what betweenness captures and degree misses.

**Closeness centrality** is the inverse of the average shortest-path distance from a node to every other node. A high-closeness node can reach everything else quickly.

**Eigenvector centrality**: a node is important if it is connected to other important nodes, not just to many nodes. The definition is recursive. PageRank (below) formalizes the same idea for directed graphs.

In [2]:
print("degree centrality:", nx.degree_centrality(G))
print("betweenness centrality:", {k: round(v, 3) for k, v in nx.betweenness_centrality(G).items()})
print("closeness centrality:", {k: round(v, 3) for k, v in nx.closeness_centrality(G).items()})
print("eigenvector centrality:", {k: round(v, 3) for k, v in nx.eigenvector_centrality(G).items()})

degree centrality: {'A': 0.5, 'B': 0.5, 'C': 0.75, 'D': 0.5, 'E': 0.25}
betweenness centrality: {'A': 0.0, 'B': 0.0, 'C': 0.667, 'D': 0.5, 'E': 0.0}
closeness centrality: {'A': 0.571, 'B': 0.571, 'C': 0.8, 'D': 0.667, 'E': 0.444}
eigenvector centrality: {'A': 0.497, 'B': 0.497, 'C': 0.604, 'D': 0.342, 'E': 0.155}


#### Community detection

**Connected components** are the simplest version: groups of nodes reachable from each other, with no edges to other groups. The toy graph is a single component. A genuinely disconnected fraud ring would be its own component, with no edges to the legitimate-account graph.

**Louvain method** finds communities inside a single connected graph. It greedily groups nodes to maximize modularity, a score comparing the density of edges inside proposed communities with what a random graph with the same degree distribution would give. A dense subgroup like {A,B,C} (a full triangle, the maximum internal density) scores as a strong community, which is the shape a tight-knit fraud ring sharing several payment methods would produce.

In [3]:
print("connected components:", list(nx.connected_components(G)))

communities = nx.community.louvain_communities(G, seed=42)
print("Louvain communities:", communities)

connected components: [{'E', 'D', 'B', 'C', 'A'}]
Louvain communities: [{'C', 'A', 'B'}, {'E', 'D'}]


#### PageRank, worked by hand

Formula: PR(p) = (1-d)/N + d * sum(PR(i)/L(i) for every node i linking to p), where d is the damping factor (0.85 is standard), N is the number of nodes and L(i) is the out-degree of node i.

Toy setup, a directed graph with 3 nodes: A links to B and C (out-degree 2), B links to C (out-degree 1), and C links to A (out-degree 1). Initial PR = 1/3 = 0.333 for every node.
```
PR(A) = 0.05 + 0.85 * (PR(C)/1) = 0.05 + 0.85*0.333 = 0.333
PR(B) = 0.05 + 0.85 * (PR(A)/2) = 0.05 + 0.85*0.1667 = 0.192
PR(C) = 0.05 + 0.85 * (PR(A)/2 + PR(B)/1) = 0.05 + 0.85*(0.1667+0.333) = 0.475
```
(Check: 0.333 + 0.192 + 0.475 = 1.0, correctly normalized.)

After one iteration C already has the highest PageRank (0.475) because both other nodes link to it. B has the lowest (0.192): only A points to it, and A splits its vote between B and C. This is the core intuition: importance flows through incoming links, and a link from a node with few outgoing links carries more weight than one from a node with many (dividing by L(i) is the "split the vote" mechanic).

In [4]:
DG = nx.DiGraph()
DG.add_edges_from([("A","B"), ("A","C"), ("B","C"), ("C","A")])

pagerank = nx.pagerank(DG, alpha=0.85)
print("PageRank (converged):", {k: round(v, 3) for k, v in pagerank.items()})

PageRank (converged): {'A': 0.388, 'B': 0.215, 'C': 0.397}


#### Relevance to fraud detection

This is the technique originally scoped as notebook 4 (`04-graph-fraud-rings.ipynb`) in the fraud-detection-agent project plan. Build a graph where nodes are accounts or transactions and edges are shared identifiers (device fingerprint, phone number, payment method, IP address). Then look for unusually dense connected components (a fraud ring sharing infrastructure) or unusually high-centrality or PageRank nodes (a hub account, possibly a mule receiving funds from many compromised accounts). A flat row-per-transaction model cannot see this pattern, because the signal lives in the connections between rows. A graph representation surfaces it, and the tabular models in `09-trees.ipynb` and `08-classical-ml.ipynb` cannot.

#### Worked example: transaction network (NetworkX)

In [5]:
import pandas as pd
import networkx as nx

# Simple transaction network: who sent money to whom
transactions = pd.DataFrame({
    'sender': ['Alice', 'Bob', 'Charlie', 'Alice', 'David', 'Eve', 'Frank'],
    'receiver': ['Bob', 'Charlie', 'David', 'Eve', 'Frank', 'Frank', 'George'],
    'amount': [100, 500, 200, 150, 300, 250, 1000]
})

print(transactions)

# RED FLAGS
# Long chain of transfers (money launderng)
# Circular transfers (suspicious loops)
# Hub Nodes (Someone receiving form many people)

G = nx.from_pandas_edgelist(
    transactions,
    source='sender',
    target='receiver',
    edge_attr='amount',
    create_using=nx.DiGraph()
)

# Basic graph info
print(f"Nodes (people): {G.nodes()}")
print(f"Edges (transactions): {G.edges()}")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print()

    sender receiver  amount
0    Alice      Bob     100
1      Bob  Charlie     500
2  Charlie    David     200
3    Alice      Eve     150
4    David    Frank     300
5      Eve    Frank     250
6    Frank   George    1000
Nodes (people): ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank', 'George']
Edges (transactions): [('Alice', 'Bob'), ('Alice', 'Eve'), ('Bob', 'Charlie'), ('Charlie', 'David'), ('David', 'Frank'), ('Eve', 'Frank'), ('Frank', 'George')]
Number of nodes: 7
Number of edges: 7



In [6]:
# Who has the most outgoing transactions (senders)?
# Out-degree: how many people someone sent money to (activity level)

out_degree = dict(G.out_degree())
print("Outgoing transactions (sender activity):")
print(sorted(out_degree.items(), key=lambda x: x[1], reverse=True))
print()

# 1. out_degree.items() — convert dict to list of [(key, value)] tuples
# 2. key=lambda x: x[1] — sort by the value (second element, index 1)

# --------- #

# Who has the most incoming transactions (receivers)?
# In-degree: how many people sent money to someone (popularity/sink)

in_degree = dict(G.in_degree())
print("Incoming transactions (receiver activity):")
print(sorted(in_degree.items(), key=lambda x: x[1], reverse=True))

Outgoing transactions (sender activity):
[('Alice', 2), ('Bob', 1), ('Charlie', 1), ('David', 1), ('Eve', 1), ('Frank', 1), ('George', 0)]

Incoming transactions (receiver activity):
[('Frank', 2), ('Bob', 1), ('Charlie', 1), ('David', 1), ('Eve', 1), ('George', 1), ('Alice', 0)]


> In fraud: high in-degree + high out-degree together = suspicious hub.

In [7]:
# Detect cycles - suspicious pattern in fraud
cycles = list(nx.simple_cycles(G))
print(f"Circular transfers detected: {len(cycles)}")
for cycle in cycles:
    print(f"  Cycle: {' → '.join(cycle)} → {cycle[0]}")
print()

# Example: if Alice → Bob → Charlie → Alice, 
# money goes in a circle (money laundering red flag)
# Cycles: Circular money flows (fraud indicator)

#-------------#

# Check if graph is acyclic (no cycles at all)
is_acyclic = nx.is_directed_acyclic_graph(G)
print(f"Is graph acyclic (no cycles)? {is_acyclic}")
print()

# Find all paths from one person to another
# Example: all ways money can flow from Alice to George
if 'Alice' in G and 'George' in G:
    paths = list(nx.all_simple_paths(G, 'Alice', 'George'))
    print(f"Paths from Alice to George:")
    for path in paths:
        print(f"  {' → '.join(path)}")

# All paths: How money can flow through the network (tracing suspicious chains)

Circular transfers detected: 0

Is graph acyclic (no cycles)? True

Paths from Alice to George:
  Alice → Bob → Charlie → David → Frank → George
  Alice → Eve → Frank → George


In [8]:
# Betweenness centrality: who sits in the middle of many paths?
# High value = person is a "middleman" (suspicious in fraud)
betweenness = nx.betweenness_centrality(G)
print("Betweenness Centrality (middleman score):")
for person, score in sorted(betweenness.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: {score:.3f}")
print()

# Closeness centrality: who is closest to everyone?
# High value = person is well-connected (hub)
closeness = nx.closeness_centrality(G)
print("Closeness Centrality (connectivity hub):")
for person, score in sorted(closeness.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: {score:.3f}")

Betweenness Centrality (middleman score):
  Frank: 0.167
  Charlie: 0.133
  David: 0.133
  Bob: 0.067
  Eve: 0.067
  Alice: 0.000
  George: 0.000

Closeness Centrality (connectivity hub):
  Frank: 0.463
  George: 0.400
  David: 0.250
  Charlie: 0.222
  Bob: 0.167
  Eve: 0.167
  Alice: 0.000


In [9]:
# PageRank: influence based on incoming connections
pagerank = nx.pagerank(G, weight='amount')
print("PageRank (influence/authority):")
for person, rank in sorted(pagerank.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: {rank:.4f}")
print()

# Who receives the most total money (weighted in-degree)?
weighted_in_degree = dict(G.in_degree(weight='amount'))
print("Weighted In-Degree (total money received):")
for person, amount in sorted(weighted_in_degree.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: ${amount}")


PageRank (influence/authority):
  George: 0.2696
  Frank: 0.2534
  David: 0.1526
  Charlie: 0.1159
  Eve: 0.0818
  Bob: 0.0726
  Alice: 0.0542

Weighted In-Degree (total money received):
  George: $1000
  Frank: $550
  Charlie: $500
  David: $200
  Eve: $150
  Bob: $100
  Alice: $0


In [10]:
# Ego graph: all people directly connected to Frank
ego_frank = nx.ego_graph(G, 'Frank')
print(f"People directly connected to Frank:")
print(f"  Nodes: {list(ego_frank.nodes())}")
print(f"  Edges: {list(ego_frank.edges())}")
print()

# Predecessors (who sent money to George)?
george_senders = list(G.predecessors('George'))
print(f"Who sent money to George? {george_senders}")
print()

# Successors (who did Frank send money to)?
frank_receivers = list(G.successors('Frank'))
print(f"Who did Frank send money to? {frank_receivers}")
print()

# Summary: create a suspicious subgraph
# (everyone who connects to George within 1 hop)
suspicious_subgraph = ego_frank.copy()
print(f"Suspicious network around Frank:")
print(f"  Involved: {suspicious_subgraph.nodes()}")
print(f"  Transactions: {suspicious_subgraph.edges()}")


People directly connected to Frank:
  Nodes: ['Frank', 'George']
  Edges: [('Frank', 'George')]

Who sent money to George? ['Frank']

Who did Frank send money to? ['George']

Suspicious network around Frank:
  Involved: ['Frank', 'George']
  Transactions: [('Frank', 'George')]


In [11]:
# Composite fraud score: combine multiple signals
fraud_score = {}

for node in G.nodes():
    score = 0
    
    # Signal 1: Betweenness (middleman activity)
    score += betweenness[node] * 100
    
    # Signal 2: High in-degree + out-degree (hub)
    in_deg = G.in_degree(node)
    out_deg = G.out_degree(node)
    if in_deg > 1 and out_deg > 1:
        score += 50
    
    # Signal 3: PageRank (influence)
    score += pagerank[node] * 50
    
    # Signal 4: Money concentration (lots in, lots out)
    in_amount = weighted_in_degree.get(node, 0)
    out_amount = dict(G.out_degree(weight='amount')).get(node, 0)
    if in_amount > 300 and out_amount > 0:
        score += 30
    
    fraud_score[node] = score

print("Fraud Suspicion Score (higher = more suspicious):")
for person, score in sorted(fraud_score.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: {score:.2f}")


Fraud Suspicion Score (higher = more suspicious):
  Frank: 59.34
  Charlie: 49.13
  David: 20.97
  George: 13.48
  Eve: 10.76
  Bob: 10.30
  Alice: 2.71


#### Clustering coefficient and density

Clustering coefficient: share of a node's neighbour pairs that are connected (triangle density) — high in tight communities/fraud rings. Connected components: maximal reachable sets (BFS/DFS/union-find); weak vs strong for directed. Density=2E/(V(V−1)) for undirected.

#### Link prediction

Predict missing/future edges. Heuristics: common neighbours, Jaccard, Adamic–Adar (shared neighbours weighted 1/log degree — rare neighbours count more), preferential attachment. Embedding: node2vec/GNN with score=f(emb_u, emb_v). Evaluate on held-out edges + sampled negatives (AUC/AP). Uses: recommendations, hidden-link fraud.

In [1]:
import networkx as nx
G = nx.karate_club_graph()
for u, v, score in nx.adamic_adar_index(G, [(0, 9), (0, 33), (5, 16)]):
    print(f"pair ({u},{v}): Adamic-Adar {score:.2f}, common neighbors {len(list(nx.common_neighbors(G, u, v)))}")

pair (0,9): Adamic-Adar 0.43, common neighbors 1
pair (0,33): Adamic-Adar 2.71, common neighbors 4
pair (5,16): Adamic-Adar 0.72, common neighbors 1


#### Distance to known-bad accounts

A node a few hops from a confirmed fraud/bad actor carries elevated risk even with no directly suspicious transaction of its own — proximity in the graph is itself a signal a purely tabular, per-transaction feature set can't represent. Computed via BFS from seed bad-nodes (unweighted) or a shortest-weighted-path algorithm if edge weights encode transaction strength/frequency.

In [2]:
import networkx as nx
from collections import Counter
G = nx.barabasi_albert_graph(200, 2, seed=1)
bad = [0, 1]                                                       # confirmed fraudulent accounts
hops = {n: min(nx.shortest_path_length(G, n, b) for b in bad) for n in G}
print("accounts by hops to the nearest known-bad account:", sorted(Counter(hops.values()).items()))

accounts by hops to the nearest known-bad account: [(0, 2), (1, 36), (2, 105), (3, 57)]


#### Graph embeddings and features for ML models

Generate many random walks starting from each node, treat each walk as a "sentence" of node IDs, then apply a Word2Vec-style skip-gram model to those walks — nodes that co-occur frequently in walks (i.e., are structurally close in the graph) end up with similar embedding vectors. Lets standard downstream ML (classification, clustering, similarity search) operate on graph structure via ordinary vector operations, without hand-engineering graph features.

#### Graph neural networks

Message passing: each node aggregates neighbour features then updates its representation; k layers = k-hop receptive field. GCN: normalized mean/sum aggregation. GraphSAGE: samples neighbours + learned aggregators ⇒ inductive, scalable. GAT: attention weights per neighbour. Issues: over-smoothing, neighbourhood explosion.

### AML Monitoring and Mule Networks

#### What AML monitoring does

Money laundering moves illicit funds through three stages: placement (getting cash into the system), layering (moving it through many accounts to hide the source) and integration (spending it as clean money). Transaction monitoring looks for the layering step. Classic systems use rules ("deposits just under $10K", "sudden volume spike"), which produce many alerts and mostly false positives, because each rule fires on a single transaction or a single account.

ML helps in two ways: it ranks alerts so analysts see the likeliest first (judged by precision at review capacity, since analysts can only clear so many a day), and it adds network context a rule on one account cannot see.

#### Mule accounts

A mule account receives money from fraud victims and passes it on quickly. Typical signals:

- High flow-through: outflow is close to inflow and leaves within hours, so the balance never builds.
- Many distinct senders paying in, and few recipients (fan-in then fan-out).
- A new account with no history, so tabular history features are thin and network features carry the signal.

Each transaction looks ordinary on its own. The pattern only exists across the account and its neighbors.

Synthetic month of payments: ordinary accounts get a salary and spend part of it, while six mule accounts receive money from 12 victims each and forward 95% to two collector accounts within hours:

In [12]:
import numpy as np, pandas as pd, networkx as nx

rng = np.random.default_rng(0)
accounts = [f"A{i}" for i in range(300)]
mules = [f"M{i}" for i in range(6)]
victims = [f"V{i}" for i in range(60)]
collectors = ["C0", "C1"]
rows = []
for a in accounts:                                          # ordinary accounts: salary in, some spending out
    for d in range(30):
        if d % 15 == 0: rows.append(("EMP", a, rng.normal(2500, 200), d + rng.random()))
        if rng.random() < 0.5: rows.append((a, f"S{rng.integers(0, 40)}", rng.gamma(2, 25), d + rng.random()))
for m in mules:                                             # mules: victims pay in, funds forwarded within hours
    for v in rng.choice(victims, 12, replace=False):
        t, amt = rng.uniform(0, 28), rng.uniform(300, 900)
        rows.append((v, m, amt, t))
        rows.append((m, rng.choice(collectors), amt * 0.95, t + rng.uniform(0.01, 0.2)))
tx = pd.DataFrame(rows, columns=["src", "dst", "amount", "t"])

inflow = tx.groupby("dst").amount.sum()
outflow = tx.groupby("src").amount.sum()
senders = tx.groupby("dst").src.nunique()
feat = pd.DataFrame({"inflow": inflow, "outflow": outflow, "n_senders": senders}).fillna(0)
feat = feat[feat.inflow > 0]
feat["flow_through"] = np.minimum(feat.outflow, feat.inflow) / feat.inflow
feat["score"] = feat.flow_through * np.log1p(feat.n_senders)
print(feat.sort_values("score", ascending=False).head(10).round(2))

       inflow  outflow  n_senders  flow_through  score
M5    6810.29  6469.77       12.0          0.95   2.44
M4    8292.38  7877.76       12.0          0.95   2.44
M3    7990.87  7591.33       12.0          0.95   2.44
M2    7282.03  6917.93       12.0          0.95   2.44
M1    8123.05  7716.90       12.0          0.95   2.44
M0    8230.11  7818.60       12.0          0.95   2.44
A15   4291.82  1093.46        1.0          0.25   0.18
A80   4821.19  1211.32        1.0          0.25   0.17
A247  4814.02  1202.38        1.0          0.25   0.17
A225  4627.37  1097.70        1.0          0.24   0.16


The six mules (`M0` to `M5`) rank at the top: almost all inflow leaves again, from a dozen distinct senders. Ordinary accounts keep most of their salary, so their flow-through is low. The next step is the network view, to find where the money ends up.

In [13]:
G = nx.DiGraph()
for r in tx.itertuples():
    if G.has_edge(r.src, r.dst): G[r.src][r.dst]["amount"] += r.amount
    else: G.add_edge(r.src, r.dst, amount=r.amount)
flagged = feat.sort_values("score", ascending=False).head(6).index
into_collectors = {c: sorted(set(G.predecessors(c)) & set(flagged)) for c in collectors}
for c, srcs in into_collectors.items():
    print(c, "receives from flagged accounts:", srcs)
print("in-degree of collectors:", {c: G.in_degree(c) for c in collectors}, "| typical ordinary account in-degree:", round(np.mean([G.in_degree(a) for a in accounts]), 2))

C0 receives from flagged accounts: ['M0', 'M1', 'M2', 'M3', 'M4', 'M5']
C1 receives from flagged accounts: ['M0', 'M1', 'M2', 'M3', 'M4', 'M5']
in-degree of collectors: {'C0': 6, 'C1': 6} | typical ordinary account in-degree: 1.0


The flagged accounts all point at the same two collectors. That shared destination is what turns six suspicious accounts into one network to investigate, and what a single-account rule would never show.

#### Why one bank is not enough

A sending bank sees its own customer's behavior, and a receiving bank sees what arrives in its accounts. Neither sees the full path of the money, which is why network-level schemes score payments for both the sending and the receiving bank (see Mastercard below), and why entity resolution (next section) matters for linking accounts, people and companies across records.

### Entity Resolution at Scale (TF-IDF / Cosine Similarity + Blocking)

Matching fragmented records of the same real-world entity (a client, a company) across naming variants — typos, abbreviations, legal-suffix differences — without exact-match or heavy embedding infrastructure.

**Why Character N-Gram TF-IDF, Not Word-Level or Exact Match**

1. TF-IDF(t,d) = tf(t,d) × log(N/df(t)) — term frequency downweighted by how common a term is across all documents.
2. Exact matching fails on naming variants; Levenshtein is character-level and fails on reordered tokens ("ABC Trading Co" vs. "Co ABC Trading"); embeddings are heavier to deploy for a problem that's mostly lexical overlap, not deep semantics.
3. Character n-grams, not word-level tokens, are the real upgrade — word-level treats a typo or abbreviation as an entirely different token, while character n-grams preserve partial overlap.
4. For entity names specifically, distinctive tokens (a rare surname, a company identifier) naturally carry more weight than common tokens ("Ltd", "Inc") purely from IDF downweighting — no extra logic needed to stop generic suffixes from dominating the match score.

In [14]:
import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)
canonical_names = [
    "Meridian Trading Company Limited", "Alpine Logistics Group Pte Ltd", "Sundara Textiles Private Limited",
    "Orion Freight Solutions Inc", "Blackwood Commodities LLC", "Northgate Manufacturing Co",
    "Vantage Export Traders Ltd", "Sapphire Marine Shipping Corp", "Everline Industrial Supplies",
    "Coral Bay Import Export Ltd",
    # Genuinely confusable near-duplicates - distinct real entities, similarly named
    "Meridian Trading Corp", "Northgate Manufacturing Group",
]

def make_variant(name, rng):
    variant = name
    choice = rng.integers(0, 5)
    if choice == 0:
        variant = variant.replace("Limited", "Ltd").replace("Company", "Co").replace("Private", "Pvt")
    elif choice == 1:
        for suf in [" Limited", " Ltd", " Pte Ltd", " Inc", " LLC", " Corp", " Co"]:
            variant = variant.replace(suf, "")
    elif choice == 2:
        i = rng.integers(0, len(variant) - 1)
        variant = variant[:i] + variant[i + 1] + variant[i] + variant[i + 2:]
    elif choice == 3:
        variant = variant.replace(" ", "  ").strip() + "."
    return variant

records = []
for entity_id, name in enumerate(canonical_names):
    for _ in range(RNG.integers(2, 5)):
        records.append({"true_entity_id": entity_id, "record_name": make_variant(name, RNG)})
records_df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"{len(records_df)} fragmented records across {len(canonical_names)} true entities")

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), lowercase=True)
tfidf_matrix = vectorizer.fit_transform(records_df["record_name"])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape} (records x character n-gram vocabulary)")

39 fragmented records across 12 true entities


TF-IDF matrix shape: (39, 642) (records x character n-gram vocabulary)


**Cosine Similarity as the Match Score**

1. cos(θ) = (A·B) / (‖A‖‖B‖) — the angle between two TF-IDF vectors.
2. Normalized so it's robust to name-length differences (a long legal name vs. a short abbreviation), unlike raw Euclidean distance — a short abbreviated name shouldn't be penalized just for having fewer character n-grams overall.

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

full_sim_matrix = cosine_similarity(tfidf_matrix)
same_entity_pair = records_df[records_df.true_entity_id == 0].index[:2]
diff_entity_pair = [records_df[records_df.true_entity_id == 0].index[0], records_df[records_df.true_entity_id == 1].index[0]]
print(f"Same-entity pair similarity: {full_sim_matrix[same_entity_pair[0], same_entity_pair[1]]:.3f}")
print(f"Different-entity pair similarity: {full_sim_matrix[diff_entity_pair[0], diff_entity_pair[1]]:.3f}")

Same-entity pair similarity: 1.000
Different-entity pair similarity: 0.048


**Blocking to Avoid O(n²) Comparisons**

1. Group candidate pairs by a coarse key first (country, registry-code prefix, first characters), then only compute full cosine similarity within blocks — cutting comparisons from all-pairs O(n²) down to a much smaller candidate set.
2. A first-2-characters key (used below for illustration) would miss true matches whose first characters differ due to a typo right at the start of the name — a real system blocks on something more stable, like a registry number prefix or normalized country code.

In [16]:
records_df["block_key"] = records_df["record_name"].str.strip().str[:2].str.lower()
candidate_pairs = []
for block_key, group in records_df.groupby("block_key"):
    idxs = group.index.tolist()
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            candidate_pairs.append((idxs[i], idxs[j]))

total_pairs_no_blocking = len(records_df) * (len(records_df) - 1) // 2
print(f"All-pairs comparisons (no blocking): {total_pairs_no_blocking}")
print(f"Candidate pairs after blocking: {len(candidate_pairs)}")
print(f"Reduction: {(1 - len(candidate_pairs) / total_pairs_no_blocking):.1%}")

All-pairs comparisons (no blocking): 741
Candidate pairs after blocking: 68
Reduction: 90.8%


**Choosing the Match Threshold**

1. Sweep candidate thresholds against known true/false match labels, favoring precision.
2. A false merge corrupts a record permanently — worse than a missed match, which just stays unresolved for manual review.
3. Genuinely confusable near-duplicate entities (e.g. "Meridian Trading Corp" vs. "...Company Limited") make this a real precision/recall tradeoff, not a formality — precision visibly drops at low thresholds.

In [17]:
pair_scores = []
for i, j in candidate_pairs:
    sim = full_sim_matrix[i, j]
    is_true_match = records_df.loc[i, "true_entity_id"] == records_df.loc[j, "true_entity_id"]
    pair_scores.append({"i": i, "j": j, "similarity": sim, "true_match": is_true_match})
pairs_df = pd.DataFrame(pair_scores)

from sklearn.metrics import precision_score, recall_score, f1_score
results = []
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = (pairs_df["similarity"] >= threshold).astype(int)
    true = pairs_df["true_match"].astype(int)
    results.append({"threshold": threshold, "precision": precision_score(true, pred, zero_division=0),
                     "recall": recall_score(true, pred, zero_division=0), "f1": f1_score(true, pred, zero_division=0)})
results_df = pd.DataFrame(results)
print(results_df.round(3))

qualifying = results_df[results_df["precision"] >= 0.90]
best_threshold = qualifying["threshold"].min() if len(qualifying) else results_df.loc[results_df["precision"].idxmax(), "threshold"]
pred_final = (pairs_df["similarity"] >= best_threshold).astype(int)
accuracy = (pred_final == pairs_df["true_match"].astype(int)).mean()
print(f"\nSelected threshold: {best_threshold}, overall accuracy at this threshold: {accuracy:.1%}")

   threshold  precision  recall     f1
0        0.3      0.706   1.000  0.828
1        0.4      0.706   1.000  0.828
2        0.5      0.727   1.000  0.842
3        0.6      0.833   0.938  0.882
4        0.7      0.930   0.833  0.879
5        0.8      1.000   0.667  0.800

Selected threshold: 0.7, overall accuracy at this threshold: 83.8%


**Handling the Borderline Band**

1. A false match merges two distinct entities — bad for downstream targeting and for compliance/KYC accuracy.
2. Borderline-similarity pairs get routed to manual review rather than auto-merged or auto-rejected — the borderline band genuinely mixes true and false matches, not cleanly separable at that similarity range, which is exactly why it needs a human decision rather than a lower automated threshold that would misclassify some fraction either direction.

In [18]:
LOWER_BAND, UPPER_BAND = best_threshold - 0.15, best_threshold + 0.15
borderline = pairs_df[(pairs_df["similarity"] >= LOWER_BAND) & (pairs_df["similarity"] < UPPER_BAND)]
print(f"Borderline pairs routed to manual review ({LOWER_BAND:.2f}-{UPPER_BAND:.2f}): {len(borderline)}")
if len(borderline):
    print(borderline[["similarity", "true_match"]].sort_values("similarity", ascending=False))

Borderline pairs routed to manual review (0.55-0.85): 34
    similarity  true_match
0     0.810873        True
33    0.797855       False
59    0.789362        True
57    0.789362        True
55    0.789362        True
43    0.776695       False
41    0.776695       False
26    0.765999        True
32    0.765999        True
6     0.757103        True
7     0.757103        True
3     0.757103        True
40    0.684428        True
22    0.656876        True
35    0.654511       False
39    0.651295        True
66    0.630935        True
50    0.630623       False
48    0.630623       False
20    0.628576       False
19    0.628576       False
42    0.626908        True
34    0.621786       False
45    0.599093       False
47    0.599093       False
37    0.597821       False
28    0.582821       False
24    0.582821       False
30    0.582821       False
23    0.582821       False
51    0.576002       False
53    0.576002       False
44    0.575419        True
49    0.552340        Tru

### Blogs

- **Mastercard, Consumer Fraud Risk.** Mastercard's service scores a payment in real time for the sending bank, and since its 2024 update also gives receiving banks an inbound risk alert within seconds, to spot payments heading to mule accounts. In initial offline tests, Mastercard reports a 60% average improvement in a bank's ability to identify high-risk mule accounts early. Mastercard says 11 UK banks have used the service since early 2023 and reports that APP fraud value fell 12% in 2023, from £389m to £341m. These are Mastercard's own figures. [Mastercard expands first-of-its-kind AI technology to help banks protect more consumers from scams in real time](https://newsroom.mastercard.com/news/press/2024/september/mastercard-expands-first-of-its-kind-ai-technology-to-help-banks-protect-more-consumers-from-scams-in-real-time/)
- **HSBC with Google Cloud AML AI.** HSBC replaced a one-size-fits-all rules system with a model trained on its own data that looks at individual accounts, criminal networks and known patterns, across about 1.2 billion transactions a month. Compared with the 12 months before go-live (excluding the first month), HSBC reports finding 2 to 4 times more confirmed suspicious activity with about 60% fewer alerts, and a shorter time to detect (8 days after the first alert). HSBC's model governance work was recognised as Celent's 2023 Model Risk Manager of the Year. [How HSBC fights money launderers with artificial intelligence](https://cloud.google.com/blog/topics/financial-services/how-hsbc-fights-money-launderers-with-artificial-intelligence)

Common pattern: score at the network and account level instead of rules on single transactions, give analysts fewer and better alerts, and measure the change against a matched before-and-after period.